## 1. Import libraries and load data


# Image Analysis and Classification: JFIF vs PPM

This notebook covers:
1. Loading and preparing the data
2. Exploratory data analysis and visualization
3. Machine-learning models for image classification
4. Model performance evaluation


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
import joblib
import warnings

from jpeg_compression_detection.paths import (
    DATA_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    display_path,
)

warnings.filterwarnings('ignore')

# Configure visualization defaults.
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline


In [36]:
# Configure plot font sizes
font_size = 20  # Default font size (adjust as needed)

In [ ]:
# Load the data
df_jfif = pd.read_csv(DATA_DIR / 'results_jfif.csv')
df_ppm = pd.read_csv(DATA_DIR / 'results_ppm.csv')

# Add labels
df_jfif['label'] = 'jfif'
df_ppm['label'] = 'ppm'

# Combine the datasets
df = pd.concat([df_jfif, df_ppm], ignore_index=True)

print(f"Total number of samples: {len(df)}")
print(f"Number of JFIF samples: {len(df_jfif)}")
print(f"Number of PPM samples: {len(df_ppm)}")
print(f"\nData shape: {df.shape}")
print(f"\nFirst rows:")
df.head()

## 2. Prepare data for modeling


In [ ]:
# Select features (excluding filename and extension)
feature_columns = ['blockiness_0', 'blockiness_1', 'blockiness_2', 'blockiness_3', 
                   'blockiness_4', 'blockiness_5', 'blockiness_6', 'blockiness_7',
                   'blockiness_max', 'blockiness_mean', 'blockiness_std',
                   'energy_mean', 'energy_std', 'estimated_quality', 'psnr']

X = df[feature_columns]
y = df['label']

# Convert labels to binary values (0 = JFIF, 1 = PPM)
y_binary = (y == 'ppm').astype(int)

# Split into training, validation, and test sets with shuffling
# First, separate the test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42, stratify=y_binary, shuffle=True)

# Then split the remaining data into training (64%) and validation (16%) sets
# 0.2 of 0.8 equals 0.16 of the full dataset
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp, shuffle=True)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nClass distribution in the training set:")
print(pd.Series(y_train).value_counts())
print(f"\nClass distribution in the validation set:")
print(pd.Series(y_val).value_counts())
print(f"\nClass distribution in the test set:")
print(pd.Series(y_test).value_counts())

## 3. Data visualization


### 3.1. Combined blockiness line plot


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

blockiness_cols = ['blockiness_0', 'blockiness_1', 'blockiness_2', 'blockiness_3', 
                   'blockiness_4', 'blockiness_5', 'blockiness_6', 'blockiness_7']

# Plot the JFIF samples
for col in blockiness_cols:
    values_jfif = df_jfif[col].values
    ax.plot(range(len(values_jfif)), values_jfif, alpha=0.5, linewidth=0.5, color='blue', label='JFIF' if col == blockiness_cols[0] else '')

# Plot the PPM samples
for col in blockiness_cols:
    values_ppm = df_ppm[col].values
    # if col != 'blockiness_0':
    #     continue
    ax.plot(range(len(values_ppm)), values_ppm, alpha=0.5, linewidth=0.5, color='red', label='PPM' if col == blockiness_cols[0] else '')

ax.set_xlabel('Sample index', fontsize=font_size)
ax.set_ylabel('Blockiness value', fontsize=font_size)
ax.set_title('Combined line plot of all blockiness components (JFIF vs PPM)', fontsize=font_size+2, fontweight='bold')
ax.legend(fontsize=font_size)
ax.grid(True, alpha=0.3)
ax.tick_params(axis='both', which='major', labelsize=font_size-2)
plt.tight_layout()
plt.show()

### 3.2. Boxplots of key indicators


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

features_to_plot = ['blockiness_mean', 'blockiness_std', 'energy_mean', 'energy_std', 'estimated_quality', 'psnr']

for i, feature in enumerate(features_to_plot):
    data_to_plot = [df_jfif[feature], df_ppm[feature]]
    bp = axes[i].boxplot(data_to_plot, labels=['JFIF', 'PPM'], patch_artist=True)
    
    # Set box colors
    colors = ['lightblue', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[i].set_title(f'{feature}', fontsize=font_size, fontweight='bold')
    axes[i].set_ylabel('Value', fontsize=font_size-2)
    axes[i].tick_params(axis='both', which='major', labelsize=font_size-2)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Boxplots comparing JFIF and PPM indicators', fontsize=font_size+4, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### 3.3. Indicator histograms


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.ravel()

# Exclude 'psnr' and 'estimated_quality' from features_to_plot
features_to_plot_filtered = [f for f in features_to_plot if f not in ['psnr', 'estimated_quality']]

# Readable feature names
display_names = {
    'blockiness_mean': 'Mean blockiness',
    'blockiness_std': 'Blockiness standard deviation',
    'energy_mean': 'Mean energy',
    'energy_std': 'Energy standard deviation'
}

for i, feature in enumerate(features_to_plot_filtered):
    # Calculate common range for both datasets
    min_val = min(df_jfif[feature].min(), df_ppm[feature].min())
    max_val = max(df_jfif[feature].max(), df_ppm[feature].max())
    hist_range = (min_val, max_val)
    
    # Calculate bin width as 2% of the plot range
    bin_width = (max_val - min_val) * 0.02
    bins = np.arange(min_val, max_val + bin_width, bin_width)
    
    axes[i].hist(df_jfif[feature], bins=bins, alpha=0.6, label='JFIF', color='blue', edgecolor='black', range=hist_range)
    axes[i].hist(df_ppm[feature], bins=bins, alpha=0.6, label='PPM', color='red', edgecolor='black', range=hist_range)
    axes[i].set_title(f'Histogram - {display_names[feature]}', fontsize=font_size+4, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=font_size+4)
    axes[i].set_ylabel('Frequency', fontsize=font_size+4)
    axes[i].legend(fontsize=font_size+4)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='both', which='major', labelsize=font_size+2)
    axes[i].locator_params(axis='x', nbins=6)
    axes[i].locator_params(axis='y', nbins=6)
    axes[i].ticklabel_format(style='plain', axis='both')
    axes[i].grid(True, which='both', linewidth=0.5, alpha=0.7, color='gray')

plt.tight_layout()
plt.show()

### 3.4. Mean blockiness components for both groups


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Compute the mean of every blockiness component
mean_jfif = [df_jfif[col].mean() for col in blockiness_cols]
mean_ppm = [df_ppm[col].mean() for col in blockiness_cols]

x_labels = ['B0', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7']

ax.plot(x_labels, mean_jfif, marker='o', linewidth=2, markersize=8, label='JFIF', color='blue')
ax.plot(x_labels, mean_ppm, marker='s', linewidth=2, markersize=8, label='PPM', color='red')

ax.set_xlabel('Blockiness component', fontsize=font_size)
ax.set_ylabel('Mean value', fontsize=font_size)
ax.set_title('Mean values of each blockiness component (JFIF vs PPM)', fontsize=font_size+2, fontweight='bold')
ax.legend(fontsize=font_size)
ax.tick_params(axis='both', which='major', labelsize=font_size-2)
ax.grid(True, alpha=0.3, color='grey')
plt.tight_layout()
plt.show()

### 3.5. Combined line plot of all indicators


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Sample every tenth row for readability
sample_rate = 10
indices = range(0, min(len(df_jfif), len(df_ppm)), sample_rate)

for feature in features_to_plot:
    values_combined = df.iloc[::sample_rate][feature].values
    ax.plot(range(len(values_combined)), values_combined, alpha=0.6, linewidth=1, label=feature)

ax.set_xlabel('Sample index (every tenth row)', fontsize=font_size)
ax.set_ylabel('Indicator value', fontsize=font_size)
ax.set_title('Combined line plot of all sampled indicators', fontsize=font_size+2, fontweight='bold')
ax.legend(loc='best', fontsize=font_size-2)
ax.tick_params(axis='both', which='major', labelsize=font_size-2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.6. Correlation matrix


In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

correlation_matrix = X.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax, annot_kws={"fontsize": font_size-8})

ax.set_title('Indicator correlation matrix', fontsize=font_size+4, fontweight='bold')
ax.tick_params(axis='both', which='major', labelsize=font_size-2)
plt.tight_layout()
plt.show()

## 4. Machine-learning models


### 4.1. Train multiple classification algorithms


In [ ]:
# Define the candidate models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

# Store model results
results = {}

# Train and evaluate the models
print("Training models...\n")
for name, model in models.items():
    print(f"Training: {name}...")
    model.fit(X_train_scaled, y_train)
    
    # Predictions on the validation set
    y_val_pred = model.predict(X_val_scaled)
    y_val_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    # Predictions on the test set
    y_test_pred = model.predict(X_test_scaled)
    y_test_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Validation metrics
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_auc_score = roc_auc_score(y_val, y_val_pred_proba)
    
    # Test metrics
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_auc_score = roc_auc_score(y_test, y_test_pred_proba)
    
    results[name] = {
        'model': model,
        'y_val_pred': y_val_pred,
        'y_val_pred_proba': y_val_pred_proba,
        'y_test_pred': y_test_pred,
        'y_test_pred_proba': y_test_pred_proba,
        'val_accuracy': val_accuracy,
        'val_auc': val_auc_score,
        'test_accuracy': test_accuracy,
        'test_auc': test_auc_score
    }
    
    print(f"  Validation - Accuracy: {val_accuracy:.4f}, AUC: {val_auc_score:.4f}")
    print(f"  Test - Accuracy: {test_accuracy:.4f}, AUC: {test_auc_score:.4f}\n")

print("Training complete.")

### 4.2. Compare model performance


In [ ]:
# Build the validation comparison table
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Val_Accuracy': [results[m]['val_accuracy'] for m in results.keys()],
    'Val_AUC': [results[m]['val_auc'] for m in results.keys()],
    'Test_Accuracy': [results[m]['test_accuracy'] for m in results.keys()],
    'Test_AUC': [results[m]['test_auc'] for m in results.keys()]
})
comparison_df = comparison_df.sort_values('Val_Accuracy', ascending=False)

print("=== MODEL COMPARISON (validation and test sets) ===\n")
print(comparison_df.to_string(index=False))
print(f"\n{'='*70}")

# Visualize the comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Validation accuracy
axes[0, 0].barh(comparison_df['Model'], comparison_df['Val_Accuracy'], color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Accuracy', fontsize=font_size)
axes[0, 0].set_title('Model accuracy comparison (validation)', fontsize=font_size+2, fontweight='bold')
axes[0, 0].set_xlim([0.9, 1.0])
axes[0, 0].tick_params(axis='both', which='major', labelsize=font_size-2)
for i, v in enumerate(comparison_df['Val_Accuracy']):
    axes[0, 0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=font_size-2)

# Validation AUC
axes[0, 1].barh(comparison_df['Model'], comparison_df['Val_AUC'], color='lightcoral', edgecolor='black')
axes[0, 1].set_xlabel('AUC', fontsize=font_size)
axes[0, 1].set_title('Model AUC comparison (validation)', fontsize=font_size+2, fontweight='bold')
axes[0, 1].set_xlim([0.9, 1.0])
axes[0, 1].tick_params(axis='both', which='major', labelsize=font_size-2)
for i, v in enumerate(comparison_df['Val_AUC']):
    axes[0, 1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=font_size-2)

# Test accuracy
axes[1, 0].barh(comparison_df['Model'], comparison_df['Test_Accuracy'], color='lightgreen', edgecolor='black')
axes[1, 0].set_xlabel('Accuracy', fontsize=font_size)
axes[1, 0].set_title('Model accuracy comparison (test)', fontsize=font_size+2, fontweight='bold')
axes[1, 0].set_xlim([0.9, 1.0])
axes[1, 0].tick_params(axis='both', which='major', labelsize=font_size-2)
for i, v in enumerate(comparison_df['Test_Accuracy']):
    axes[1, 0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=font_size-2)

# Test AUC
axes[1, 1].barh(comparison_df['Model'], comparison_df['Test_AUC'], color='lightyellow', edgecolor='black')
axes[1, 1].set_xlabel('AUC', fontsize=font_size)
axes[1, 1].set_title('Model AUC comparison (test)', fontsize=font_size+2, fontweight='bold')
axes[1, 1].set_xlim([0.9, 1.0])
axes[1, 1].tick_params(axis='both', which='major', labelsize=font_size-2)
for i, v in enumerate(comparison_df['Test_AUC']):
    axes[1, 1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=font_size-2)

plt.tight_layout()
plt.show()

### 4.3. Detailed analysis of the best model


In [ ]:
# Select the best model using validation performance
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = results[best_model_name]

print(f"=== BEST MODEL: {best_model_name} ===\n")
print(f"Validation - Accuracy: {best_model_results['val_accuracy']:.4f}, AUC: {best_model_results['val_auc']:.4f}")
print(f"Test - Accuracy: {best_model_results['test_accuracy']:.4f}, AUC: {best_model_results['test_auc']:.4f}\n")

# Classification reports for validation and test sets
print("=== Classification Report (validation) ===")
print(classification_report(y_val, best_model_results['y_val_pred'], 
                          target_names=['JFIF (0)', 'PPM (1)']))

print("\n=== Classification Report (test) ===")
print(classification_report(y_test, best_model_results['y_test_pred'], 
                          target_names=['JFIF (0)', 'PPM (1)']))

# Confusion matrices for both sets
cm_val = confusion_matrix(y_val, best_model_results['y_val_pred'])
cm_test = confusion_matrix(y_test, best_model_results['y_test_pred'])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Validation confusion matrix
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], 
            xticklabels=['JFIF', 'PPM'], yticklabels=['JFIF', 'PPM'], annot_kws={"fontsize": font_size})
axes[0, 0].set_xlabel('Prediction', fontsize=font_size)
axes[0, 0].set_ylabel('True class', fontsize=font_size)
axes[0, 0].set_title(f'Confusion matrix - {best_model_name} (validation)', fontsize=font_size+2, fontweight='bold')
axes[0, 0].tick_params(axis='both', which='major', labelsize=font_size-2)

# Test confusion matrix
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', ax=axes[0, 1], 
            xticklabels=['JFIF', 'PPM'], yticklabels=['JFIF', 'PPM'], annot_kws={"fontsize": font_size})
axes[0, 1].set_xlabel('Prediction', fontsize=font_size)
axes[0, 1].set_ylabel('True class', fontsize=font_size)
axes[0, 1].set_title(f'Confusion matrix - {best_model_name} (Test)', fontsize=font_size+2, fontweight='bold')
axes[0, 1].tick_params(axis='both', which='major', labelsize=font_size-2)

# Validation ROC curve
fpr_val, tpr_val, _ = roc_curve(y_val, best_model_results['y_val_pred_proba'])
axes[1, 0].plot(fpr_val, tpr_val, linewidth=2, label=f'{best_model_name} (AUC = {best_model_results["val_auc"]:.4f})')
axes[1, 0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
axes[1, 0].set_xlabel('False Positive Rate', fontsize=font_size)
axes[1, 0].set_ylabel('True Positive Rate', fontsize=font_size)
axes[1, 0].set_title('ROC curve (validation)', fontsize=font_size+2, fontweight='bold')
axes[1, 0].legend(fontsize=font_size)
axes[1, 0].tick_params(axis='both', which='major', labelsize=font_size-2)
axes[1, 0].grid(True, alpha=0.3)

# Test ROC curve
fpr_test, tpr_test, _ = roc_curve(y_test, best_model_results['y_test_pred_proba'])
axes[1, 1].plot(fpr_test, tpr_test, linewidth=2, label=f'{best_model_name} (AUC = {best_model_results["test_auc"]:.4f})', color='green')
axes[1, 1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
axes[1, 1].set_xlabel('False Positive Rate', fontsize=font_size)
axes[1, 1].set_ylabel('True Positive Rate', fontsize=font_size)
axes[1, 1].set_title('ROC curve (test)', fontsize=font_size+2, fontweight='bold')
axes[1, 1].legend(fontsize=font_size)
axes[1, 1].tick_params(axis='both', which='major', labelsize=font_size-2)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.4. Feature importance for tree-based models


In [ ]:
# Analyze feature importance for tree-based models
tree_based_models = ['Random Forest', 'Gradient Boosting']

for model_name in tree_based_models:
    if model_name in results:
        model = results[model_name]['model']
        
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            indices = np.argsort(importances)[::-1]
            
            plt.figure(figsize=(12, 6))
            plt.barh(range(len(importances)), importances[indices], color='teal', edgecolor='black')
            plt.yticks(range(len(importances)), [feature_columns[i] for i in indices], fontsize=font_size-2)
            plt.xlabel('Feature importance', fontsize=font_size)
            plt.title(f'Feature importance - {model_name}', fontsize=font_size+2, fontweight='bold')
            plt.gca().invert_yaxis()
            plt.tick_params(axis='both', which='major', labelsize=font_size-2)
            plt.grid(True, alpha=0.3, axis='x')
            plt.tight_layout()
            plt.show()
            
            print(f"\n=== Top 5 features for {model_name} ===")
            for i in range(min(5, len(importances))):
                idx = indices[i]
                print(f"{i+1}. {feature_columns[idx]}: {importances[idx]:.4f}")

### 4.5. Save the best model


In [ ]:
# Save the best model and its preprocessing files.
MODELS_DIR.mkdir(parents=True, exist_ok=True)

best_model = results[best_model_name]['model']
model_path = MODELS_DIR / f'best_model_{best_model_name.replace(" ", "_").lower()}.pkl'
joblib.dump(best_model, model_path)
print(f"Model '{best_model_name}' saved to: {display_path(model_path)}")

# Also save a stable alias so downstream evaluation does not depend on the winner's name.
best_model_alias_path = MODELS_DIR / 'best_classification_model.pkl'
joblib.dump(best_model, best_model_alias_path)
print(f"Validation-selected model alias saved to: {display_path(best_model_alias_path)}")

scaler_path = MODELS_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to: {display_path(scaler_path)}")

features_path = MODELS_DIR / 'feature_columns.pkl'
joblib.dump(feature_columns, features_path)
print(f"Feature list saved to: {display_path(features_path)}")

all_models_path = MODELS_DIR / 'all_models.pkl'
all_models = {model_name: results[model_name]['model'] for model_name in results}
joblib.dump(all_models, all_models_path)
print(f"All models saved to: {display_path(all_models_path)}")

print("\n" + "=" * 70)
print("Models saved successfully.")
print("=" * 70)
print("\nTo load the saved files in another script, use:")
print(f"  model = joblib.load({display_path(model_path)!r})")
print(f"  scaler = joblib.load({display_path(scaler_path)!r})")
print(f"  feature_columns = joblib.load({display_path(features_path)!r})")


## 5. Prediction function for new images


In [ ]:
def classify_image(image_features, model_name='Random Forest'):
    """
    Classify an image from its quality indicators.
    
    Parameters:
    -----------
    image_features : dict or pd.Series
        Dictionary or Series containing the image indicators
    model_name : str
        Name of the model to use
    
    Returns:
    --------
    dict : Classification result containing the label and confidence
    """
    # Convert the input to a DataFrame
    if isinstance(image_features, dict):
        features_df = pd.DataFrame([image_features])
    else:
        features_df = pd.DataFrame([image_features])
    
    # Ensure that every required feature is present
    for col in feature_columns:
        if col not in features_df.columns:
            raise ValueError(f"Missing feature: {col}")
    
    # Select only the required features in the expected order
    features_df = features_df[feature_columns]
    
    # Standardize the features
    features_scaled = scaler.transform(features_df)
    
    # Select the model
    if model_name not in results:
        print(f"Model '{model_name}' was not found. Using the best model instead.")
        model_name = comparison_df.iloc[0]['Model']
    
    model = results[model_name]['model']
    
    # Prediction
    prediction = model.predict(features_scaled)[0]
    probabilities = model.predict_proba(features_scaled)[0]
    # print(model.decision_function(features_scaled)) if hasattr(model, 'decision_function') else None
    
    # Interpret the result
    label = 'PPM' if prediction == 1 else 'JFIF'
    confidence = probabilities[prediction] * 100
    
    result = {
        'label': label,
        'confidence': confidence,
        'probabilities': {
            'JFIF': probabilities[0] * 100,
            'PPM': probabilities[1] * 100
        },
        'model_used': model_name
    }
    
    return result

# Example usage
print("=== CLASSIFICATION FUNCTION READY ===\n")
print("Usage: classify_image(image_features, model_name='Random Forest')")
print("\nimage_features must contain the following indicators:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i}. {col}")

### 5.1. Test the classification function


In [ ]:
# Test random samples from the test set
print("=== CLASSIFICATION FUNCTION TESTS ===\n")

# Test 1: random JFIF image
jfif_sample = df_jfif.sample(1, random_state=42).iloc[0]
test_features_jfif = jfif_sample[feature_columns].to_dict()

result_jfif = classify_image(test_features_jfif, best_model_name)
print(f"Test 1 - True class: JFIF")
print(f"Prediction: {result_jfif['label']}")
print(f"Confidence: {result_jfif['confidence']:.2f}%")
print(f"Probabilities: JFIF={result_jfif['probabilities']['JFIF']:.2f}%, PPM={result_jfif['probabilities']['PPM']:.2f}%")
print(f"Model: {result_jfif['model_used']}\n")

# Test 2: random PPM image
ppm_sample = df_ppm.sample(1, random_state=42).iloc[0]
test_features_ppm = ppm_sample[feature_columns].to_dict()

result_ppm = classify_image(test_features_ppm, best_model_name)
print(f"Test 2 - True class: PPM")
print(f"Prediction: {result_ppm['label']}")
print(f"Confidence: {result_ppm['confidence']:.2f}%")
print(f"Probabilities: JFIF={result_ppm['probabilities']['JFIF']:.2f}%, PPM={result_ppm['probabilities']['PPM']:.2f}%")
print(f"Model: {result_ppm['model_used']}\n")

# Test 3: several samples
print("Test 3 - classify 10 random images:")
print("-" * 80)
test_samples = df.sample(10, random_state=42)

for idx, row in test_samples.iterrows():
    true_label = row['label'].upper()
    features = row[feature_columns].to_dict()
    result = classify_image(features, best_model_name)
    
    status = "✓" if result['label'] == true_label else "✗"
    print(f"{status} True: {true_label:4s} | Prediction: {result['label']:4s} | Confidence: {result['confidence']:6.2f}%")

## 5.2. Single-image tests


In [ ]:
from jpeg_compression_detection import detect_jpeg_compression

example_path = DATA_DIR / 'examples' / 'moje (6).jpg'

if not example_path.exists():
    print(f"Skipping optional example because it is not available: {display_path(example_path)}")
else:
    features = detect_jpeg_compression(example_path)
    if not isinstance(features, dict):
        print("The image indicators could not be computed.")
    else:
        print(features)

        result_single = classify_image(features, best_model_name)
        print(f"Prediction: {result_single['label']}")
        print(f"Confidence: {result_single['confidence']:.2f}%")
        print(
            f"Probabilities: JFIF={result_single['probabilities']['JFIF']:.2f}%, "
            f"PPM={result_single['probabilities']['PPM']:.2f}%"
        )
        print(f"Model: {result_single['model_used']}\n")

        print("\n=== CLASSIFICATION WITH EVERY MODEL ===\n")
        for model_name in results:
            result = classify_image(features, model_name)
            print(
                f"{model_name:25s} | Prediction: {result['label']:4s} | "
                f"Confidence: {result['confidence']:6.2f}% | "
                f"JFIF: {result['probabilities']['JFIF']:6.2f}% | "
                f"PPM: {result['probabilities']['PPM']:6.2f}%"
            )


## 5.3. Test a small sample from data/examples


In [92]:
# Optional batch test for images placed in data/examples.
# from jpeg_compression_detection import detect_jpeg_compression
#
# examples_dir = DATA_DIR / 'examples'
# image_files = [path for path in examples_dir.iterdir() if path.is_file()]
# for image_path in image_files:
#     print(f"\nAnalyzing: {display_path(image_path)}")
#     try:
#         features = detect_jpeg_compression(image_path)
#         result_single = classify_image(features, best_model_name)
#         print(f"Prediction: {result_single['label']}")
#         print(f"Confidence: {result_single['confidence']:.2f}%")
#         print(
#             f"Probabilities: JFIF={result_single['probabilities']['JFIF']:.2f}%, "
#             f"PPM={result_single['probabilities']['PPM']:.2f}%"
#         )
#         print(f"Model: {result_single['model_used']}\n")
#     except Exception as error:
#         print(f"Error processing {display_path(image_path)}: {error}")


## 6. Compare all models using ROC curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

colors = ['blue', 'red', 'green', 'orange', 'purple']

# Validation ROC curves
for (name, result), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_val, result['y_val_pred_proba'])
    ax1.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {result["val_auc"]:.4f})', color=color)

ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax1.set_xlabel('False Positive Rate', fontsize=font_size)
ax1.set_ylabel('True Positive Rate', fontsize=font_size)
ax1.set_title('ROC curve comparison for all models (validation)', fontsize=font_size+4, fontweight='bold')
ax1.legend(fontsize=font_size-1, loc='lower right')
ax1.tick_params(axis='both', which='major', labelsize=font_size-2)
ax1.grid(True, alpha=0.3)

# Test ROC curves
for (name, result), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, result['y_test_pred_proba'])
    ax2.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {result["test_auc"]:.4f})', color=color)

ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax2.set_xlabel('False Positive Rate', fontsize=font_size)
ax2.set_ylabel('True Positive Rate', fontsize=font_size)
ax2.set_title('ROC curve comparison for all models (test)', fontsize=font_size+4, fontweight='bold')
ax2.legend(fontsize=font_size-1, loc='lower right')
ax2.tick_params(axis='both', which='major', labelsize=font_size-2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Example: load a saved model


In [94]:
# Example: load the saved files in another script.
# loaded_model = joblib.load(MODELS_DIR / 'best_model_random_forest.pkl')
# loaded_scaler = joblib.load(MODELS_DIR / 'scaler.pkl')
# loaded_feature_columns = joblib.load(MODELS_DIR / 'feature_columns.pkl')
#
# print("Model loaded successfully.")
# print(f"Model type: {type(loaded_model).__name__}")
# print(f"Number of features: {len(loaded_feature_columns)}")
#
# test_sample = X_test.iloc[0:1]
# test_sample_scaled = loaded_scaler.transform(test_sample)
# prediction = loaded_model.predict(test_sample_scaled)[0]
# probabilities = loaded_model.predict_proba(test_sample_scaled)[0]
#
# print(f"\nPrediction: {'PPM' if prediction == 1 else 'JFIF'}")
# print(f"Probabilities: JFIF={probabilities[0]*100:.2f}%, PPM={probabilities[1]*100:.2f}%")
